In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from gradient_AD.gradient_descent import gradient_descent


In [2]:
RANDOM_STATE = 42

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Сохраняю столбец id из тестовой выборки для итогового submission.csv
test_id = test['id']  

## Очистка данных

1. Удаление лишних колонок.
2. Заполнение пропусков средним значением в целевой колонке avg_salary.
3. Удаление остальных строк с пропусками.

In [3]:
def drop_cols_and_fill(to_drop, df:pd.DataFrame):
    df = df.drop(to_drop, axis=1) 

    if "avg_salary" in df.columns:
        df["avg_salary"] = df["avg_salary"].fillna(df["avg_salary"].mean())
    
    df = df.dropna()
    
    return df

to_drop = ["id", "posting_date", "Company Name", "Job Description", "Job Title",]

train = drop_cols_and_fill(to_drop=to_drop, df=train)
test = drop_cols_and_fill(to_drop=to_drop, df=test)
print(train.shape)
print(test.shape)

(583, 11)
(148, 10)


## Кодирование категориальных признаков (One-Hot Encoding)

Для работы линейной регрессии необходимо преобразовать текстовые категории в числовой формат. Я использую метод **One-Hot Encoding (OHE)**, который создает бинарные признаки.

**Принцип работы:**
* Для каждого уникального значения в текстовой колонке создается отдельный столбец.
* Значение `1` ставится, если объект относится к данной категории, и `0` - в остальных случаях.

>**Примечание:** Этот метод значительно увеличивает размерность данных (количество колонок), что требует внимательного подхода к масштабированию и регуляризации модели.


In [4]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

obj_cols = train.select_dtypes(include=["object"]).columns.tolist()
encoder.fit(train[obj_cols])

def apply_ohe(df, encoder, obj_cols):
    # transform превращает данные в числовую матрицу из 1/0
    ohe_arr = encoder.transform(df[obj_cols])
    ohe_df = pd.DataFrame(ohe_arr, columns=encoder.get_feature_names_out(obj_cols), index=df.index)
    num_part = df.drop(obj_cols, axis=1)
    return pd.concat([num_part, ohe_df], axis=1)

X_train = apply_ohe(train, encoder, obj_cols)
X_test = apply_ohe(test, encoder, obj_cols)

In [5]:
# у содержит только целевую переменную avg_salary для каждого объекта.
y_train = X_train["avg_salary"]

y_log = np.log1p(y_train)

# Игнорируем avg_salary, т.к. настройка весов модели происходит относительно других признаков.
X_train = X_train.drop("avg_salary", axis=1)

# Размеры получившихся частей.
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)

X_train: (583, 575)
X_test: (148, 575)
y_train: (583,)


## Масштабирование данных и обучение модели

Для корректной работы градиентного спуска признаки приводятся к единому масштабу с помощью **Z-score нормализации** (стандартизации). Это гарантирует, что каждый признак вносит вклад в итоговый результат пропорционально своей значимости, а не величине абсолютных значений.

**Математическая модель:**
$$z = \frac{x - \mu}{\sigma}$$

Где:
*   $\mu$ - среднее значение признака (mean);
*   $\sigma$ - стандартное отклонение (standard deviation).

### Метод оптимизации
Обучение модели выполняется с помощью **градиентного спуска**, реализованного через **Forward Mode Automatic Differentiation** (автоматическое дифференцирование на дуальных числах). Этот подход позволяет вычислять точные значения производных функции потерь без использования численного приближения.


In [6]:
# Находим среднее значение и стандартное отклонение
X_mean, X_std = np.mean(X_train, axis=0), np.std(X_train, axis=0)
y_mean, y_std = y_log.mean(), y_log.std()

# Чтобы не делить на ноль, если признак константный
X_std[X_std == 0] = 1 

# Масштабируем выборку X_train и таргет y_train через Z-score нормализацию
X_train_scaled = (X_train - X_mean) / X_std
y_train_scaled = (y_log - y_mean) / y_std

# Обучаем модель, предварительно верифицировав формат данных
X_train_np = X_train_scaled.values if hasattr(X_train_scaled, 'values') else X_train_scaled
y_train_np = y_train_scaled.values if hasattr(y_train_scaled, 'values') else y_train_scaled

w, b, mse = gradient_descent(X_train_np, y_train_np, rate=0.001, n_iter=600)

print(f"MSE на обучающей выборке: {mse}")


MSE на обучающей выборке: 0.2005785463118592


In [7]:
#Масштабируем тестовую выборку по параметрам X_train
X_test_scaled = (X_test - X_mean) / X_std 
X_test_np = X_test_scaled.values if hasattr(X_test_scaled, 'values') else X_test_scaled

# Предсказание в масштабе
y_pred_scaled = X_test_np @ w + b

# Обратный Z-score (получаем логарифм)
y_pred_log = y_pred_scaled * y_std + y_mean

# Обратный логарифм (получаем реальную зарплату)
y_pred_final = np.expm1(y_pred_log)

In [8]:
# Создаем DataFrame для сохранения
submission = pd.DataFrame({
    "id": test_id,
    "avg_salary": y_pred_final
})

# Сохраняем без индексов pandas
submission.to_csv("submission.csv", index=False)

print("Файл submission.csv сохранен")

Файл submission.csv сохранен
